In [1]:
# Connect to the local SQLite database and create a text search index for the FAQ data.

from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

sqlite_index.count()

84

In [2]:
# Search for the given question in the index - top 5 results

results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

In [3]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

assistant = RAGBase(
    index=sqlite_index, # use the sqlitesarch index instead of in-memory one (we just swap the index)
    llm_client=openai_client,
)

In [4]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, you need to submit your project while submissions are still being accepted.


In [7]:
# Search fails to find right anwser because of the typo
# Answer is still generated from LLM knowledge but it's not "grounded" (based on context)
q = "How do I run Olama?"
search_results = assistant.search(q)
prompt = assistant.build_prompt(q, search_results)
answer = assistant.llm(prompt)
# answer = assistant.rag(q)
print(answer)

To run Ollama:

1. Install it from https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a model in your terminal:
   ```bash
   ollama run llama3
   ```

3. Optionally test the local server:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

If you meant something else by “Olama,” I don’t know.


In [20]:
# Without the typo, the search finds the anwser based on FAQ
q = "How do I install and run Ollama?"
# search_results = assistant.search(q)
# prompt = assistant.build_prompt(q, search_results)
# answer = assistant.llm(prompt)
answer = assistant.rag(q)
print(answer)

To install and run Ollama:

1. Go to https://ollama.com/download and choose your OS:
   - macOS: download the `.pkg` and install it
   - Windows: download the `.msi` and install it
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that Ollama is running:
   ```bash
   curl http://localhost:11434
   ```

If you want to use it from Python, install the client with:
```bash
pip install ollama
```


In [5]:
sqlite_index.close()